# L1 based feature selection
This version is split into small cells for easier reading.

In [2]:
import csv
from pathlib import Path

import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier


In [3]:
# Find project root (works from project root or notebooks folder)
def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'data' / 'processed' / 'Datasets').exists():
            return candidate
    raise FileNotFoundError('Could not find project root containing data/processed/Datasets')

PROJECT_ROOT = find_project_root(Path.cwd())
DATASETS_ROOT = PROJECT_ROOT / 'data' / 'processed' / 'Datasets'
TARGET_COLUMN = 'Label'
RANDOM_STATE = 42

print('PROJECT_ROOT:', PROJECT_ROOT)
print('DATASETS_ROOT:', DATASETS_ROOT)


PROJECT_ROOT: /Users/sultanarazia/Desktop/UNB/Machine Learning/Project/L1-feature-selection
DATASETS_ROOT: /Users/sultanarazia/Desktop/UNB/Machine Learning/Project/L1-feature-selection/data/processed/Datasets


In [4]:
# 5 required classifiers
classifiers = {
    'SVM': SVC(),
    'kNN': KNeighborsClassifier(),
    'DecisionTree': DecisionTreeClassifier(random_state=RANDOM_STATE),
    'RandomForest': RandomForestClassifier(random_state=RANDOM_STATE),
    'MLP': MLPClassifier(random_state=RANDOM_STATE, max_iter=1000),
}

# required outer CV
outer_cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)

print('Classifiers:', list(classifiers.keys()))
print('Outer CV:', outer_cv)


Classifiers: ['SVM', 'kNN', 'DecisionTree', 'RandomForest', 'MLP']
Outer CV: StratifiedKFold(n_splits=10, random_state=42, shuffle=True)


In [5]:
def load_train_dataset(dataset_id, datasets_root=DATASETS_ROOT, target_column=TARGET_COLUMN):
    train_csv = datasets_root / str(dataset_id) / 'train.csv'
    if not train_csv.exists():
        raise FileNotFoundError(f'Missing file: {train_csv}')

    with train_csv.open(newline='') as f:
        reader = csv.DictReader(f)
        if reader.fieldnames is None:
            raise ValueError(f'No header found in {train_csv}')
        if target_column not in reader.fieldnames:
            raise ValueError(f"Target column '{target_column}' not found in {train_csv}")

        feature_names = [c for c in reader.fieldnames if c != target_column]
        rows = list(reader)

    def to_float_or_nan(value):
        if value is None:
            return np.nan
        value = value.strip()
        return np.nan if value == '' else float(value)

    X = np.array([[to_float_or_nan(r[c]) for c in feature_names] for r in rows], dtype=float)
    y = np.array([r[target_column] for r in rows])
    return X, y, feature_names


In [6]:
# Reusable loop for datasets 1..16
experiment_state = {}

for dataset_id in range(1, 17):
    X, y, feature_names = load_train_dataset(dataset_id)
    folds = list(outer_cv.split(X, y))

    experiment_state[dataset_id] = {
        'X_shape': X.shape,
        'y_shape': y.shape,
        'feature_names': feature_names,
        'classifiers': classifiers,
        'outer_cv': outer_cv,
        'folds': folds,
    }

for dataset_id, info in experiment_state.items():
    print(f"Dataset {dataset_id}: X={info['X_shape']}, y={info['y_shape']}, features={len(info['feature_names'])}, folds={len(info['folds'])}")


/Users/sultanarazia/miniconda3/envs/ml-env/lib/python3.10/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 6 members, which is less than n_splits=10.
  warnings.warn(
/Users/sultanarazia/miniconda3/envs/ml-env/lib/python3.10/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 8 members, which is less than n_splits=10.
  warnings.warn(
/Users/sultanarazia/miniconda3/envs/ml-env/lib/python3.10/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 8 members, which is less than n_splits=10.
  warnings.warn(


Dataset 1: X=(9583, 19), y=(9583,), features=19, folds=10
Dataset 2: X=(9583, 19), y=(9583,), features=19, folds=10
Dataset 3: X=(9583, 19), y=(9583,), features=19, folds=10
Dataset 4: X=(9583, 19), y=(9583,), features=19, folds=10
Dataset 5: X=(9583, 19), y=(9583,), features=19, folds=10
Dataset 6: X=(9583, 19), y=(9583,), features=19, folds=10
Dataset 7: X=(9583, 19), y=(9583,), features=19, folds=10
Dataset 8: X=(9583, 19), y=(9583,), features=19, folds=10
Dataset 9: X=(8330, 265), y=(8330,), features=265, folds=10
Dataset 10: X=(8330, 265), y=(8330,), features=265, folds=10
Dataset 11: X=(8330, 265), y=(8330,), features=265, folds=10
Dataset 12: X=(8330, 265), y=(8330,), features=265, folds=10
Dataset 13: X=(8330, 265), y=(8330,), features=265, folds=10
Dataset 14: X=(8330, 265), y=(8330,), features=265, folds=10
Dataset 15: X=(8330, 265), y=(8330,), features=265, folds=10
Dataset 16: X=(8330, 265), y=(8330,), features=265, folds=10


In [7]:
# Phase 1 (Baseline): train/evaluate 5 classifiers inside outer CV
from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

def build_baseline_pipeline(name, model):
    # Imputer is required because datasets 1..8 contain blank values -> NaN
    steps = [('imputer', SimpleImputer(strategy='median'))]

    # Scale distance/gradient-based models only
    if name in ['SVM', 'kNN', 'MLP']:
        steps.append(('scaler', StandardScaler()))

    steps.append(('clf', clone(model)))
    return Pipeline(steps)

baseline_results = {}

for dataset_id in range(1, 17):
    X, y, feature_names = load_train_dataset(dataset_id)

    baseline_results[dataset_id] = {}
    for clf_name, clf_model in classifiers.items():
        fold_rows = []

        for fold_idx, (train_idx, valid_idx) in enumerate(outer_cv.split(X, y), start=1):
            X_train, X_valid = X[train_idx], X[valid_idx]
            y_train, y_valid = y[train_idx], y[valid_idx]

            pipeline = build_baseline_pipeline(clf_name, clf_model)
            pipeline.fit(X_train, y_train)
            y_pred = pipeline.predict(X_valid)

            fold_rows.append({
                'fold': fold_idx,
                'accuracy': accuracy_score(y_valid, y_pred),
                'macro_f1': f1_score(y_valid, y_pred, average='macro'),
            })

        baseline_results[dataset_id][clf_name] = fold_rows

# Quick summary (mean over 10 folds)
for dataset_id in range(1, 17):
    print(f'\nDataset {dataset_id}')
    for clf_name in classifiers.keys():
        rows = baseline_results[dataset_id][clf_name]
        acc_mean = np.mean([r['accuracy'] for r in rows])
        f1_mean = np.mean([r['macro_f1'] for r in rows])
        print(f'  {clf_name:12s} | acc={acc_mean:.4f} | macro_f1={f1_mean:.4f}')


/Users/sultanarazia/miniconda3/envs/ml-env/lib/python3.10/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 6 members, which is less than n_splits=10.
  warnings.warn(
/Users/sultanarazia/miniconda3/envs/ml-env/lib/python3.10/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 6 members, which is less than n_splits=10.
  warnings.warn(
/Users/sultanarazia/miniconda3/envs/ml-env/lib/python3.10/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 6 members, which is less than n_splits=10.
  warnings.warn(
/Users/sultanarazia/miniconda3/envs/ml-env/lib/python3.10/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 6 members, which is less than n_splits=10.
  warnings.warn(
/Users/sultanarazia/miniconda3/envs/ml-env/lib/python3.10/site-packages/sklearn/model_selection/_split.p


Dataset 1
  SVM          | acc=0.9518 | macro_f1=0.7803
  kNN          | acc=0.9784 | macro_f1=0.8133
  DecisionTree | acc=1.0000 | macro_f1=1.0000
  RandomForest | acc=0.9996 | macro_f1=0.9996
  MLP          | acc=0.9900 | macro_f1=0.9605

Dataset 2
  SVM          | acc=0.9299 | macro_f1=0.4359
  kNN          | acc=0.9523 | macro_f1=0.5718
  DecisionTree | acc=0.9599 | macro_f1=0.6059
  RandomForest | acc=0.9619 | macro_f1=0.5963
  MLP          | acc=0.9505 | macro_f1=0.5241

Dataset 3
  SVM          | acc=0.8706 | macro_f1=0.4658
  kNN          | acc=0.8900 | macro_f1=0.5800
  DecisionTree | acc=0.8900 | macro_f1=0.5800
  RandomForest | acc=0.8957 | macro_f1=0.5940
  MLP          | acc=0.8897 | macro_f1=0.5583

Dataset 4
  SVM          | acc=0.8577 | macro_f1=0.3813
  kNN          | acc=0.8646 | macro_f1=0.4801
  DecisionTree | acc=0.8667 | macro_f1=0.5051
  RandomForest | acc=0.8681 | macro_f1=0.5034
  MLP          | acc=0.8629 | macro_f1=0.4429

Dataset 5
  SVM          | acc=0.94